## Perplexity Sonar, Search, and Agent API behavior experiments.

### Sonar API via `requests` (chat completions endpoint)

Calls the Sonar chat-completions endpoint directly with the `requests` library. Includes error handling for a missing API key and for a 401 (authorization/credit) failure.

Available models: `sonar`, `sonar-pro`, `sonar-deep-research`, `sonar-reasoning-pro`.

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("PERPLEXITY_API_KEY")

if not api_key:
    raise ValueError("PERPLEXITY_API_KEY environment variable not set")

url = "https://api.perplexity.ai/chat/completions"
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
data = {
    "model": "sonar",
    "messages": [{"role": "user", "content": "What is day of week for 2027-06-01?"}],
    "max_tokens": 50,
}

response = requests.post(url, headers=headers, json=data)
if response.status_code == 200:
    result = response.json()
    print(f"Response ID: {result.get('id', 'N/A')}")
    print(result["choices"][0]["message"]["content"])
elif response.status_code == 401:
    print("No content returned - check credit balance")
else:
    print(f"Error: {response.status_code} - {response.text}")

### Sonar API via the `perplexity` SDK

Same chat-completions endpoint as above, this time using the official `perplexity` Python client instead of raw HTTP requests.

In [ ]:
from perplexity import Perplexity

client = Perplexity()

completion = client.chat.completions.create(
    model="sonar",
    messages=[
        {
            "role": "user",
            "content": "In 1 sentence, what were the results of the 2025 French Open Finals?",
        }
    ],
)

print(completion.choices[0].message.content)

### Search API

Basic test of Perplexity's Search API via the `perplexity` client. A `401 Authorization Required` error here usually means the account has no credit rather than an invalid key.

In [ ]:
from perplexity import Perplexity

client = Perplexity()

search = client.search.create(query="Perplexity AI model names latest", max_results=1)

for result in search.results:
    print(f"{result.title}: {result.url}")

### Agent API via the OpenAI SDK (`gpt` model)

Perplexity's Agent API (`https://api.perplexity.ai/v1`) is OpenAI-compatible, so it can be called with the standard OpenAI SDK by pointing `base_url` at Perplexity and authenticating with the Perplexity API key. This example requests a `gpt` model — the call goes through the OpenAI SDK's `responses` interface, but is billed to the Perplexity account, not an OpenAI account.

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()
perplex_api_key = os.getenv("PERPLEXITY_API_KEY")

client = OpenAI(api_key=perplex_api_key, base_url="https://api.perplexity.ai/v1")

try:
    response = client.responses.create(
        model="openai/gpt-5-mini",
        input="Write a one-sentence story about Hansel and Gretel",
    )
    print(response.output_text)
except Exception as e:
    print(f"API request failed ({type(e).__name__}): {e}")